In [17]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\Minas"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "Minas do Sol - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "Minas do Sol - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "Minas do Sol - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Minas do Sol", rewrite=True)
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Minas do Sol already exists
Removed existing data for project: PV Minas do Sol
Successfully loaded project
Loaded 50072 entities for project: PV Minas do Sol
Don't forget to close connection.
Data Loaded
DuckDB Loaded


CABLE AND STRUCTURE DATA
=

In [18]:
data = conn.sql("SELECT name, layer, X, Y, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Minas do Sol'")
print(data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM data
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌─────────────────────────┬─────────┬───────────┬───────────┬─────────┬─────────┬───────┬───────┬────────┐
│          name           │  layer  │     X     │     Y     │ start_x │ start_y │ end_x │ end_y │ length │
│         varchar         │ varchar │   float   │   float   │  float  │  float  │ float │ float │ float  │
├─────────────────────────┼─────────┼───────────┼───────────┼─────────┼─────────┼───────┼───────┼────────┤
│ String_Tracker_1X174p_1 │ MDS_1.1 │  511819.7 │ 8087402.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_2 │ MDS_1.1 │  511819.7 │ 8087368.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_3 │ MDS_1.1 │  511819.7 │ 8087334.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_1 │ MDS_1.1 │  511819.7 │ 8087230.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_2 │ MDS_1.1 │  511819.7 │ 8087264.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_3 │ MDS_1.1 │

STRUCTURE DATA

In [19]:
structures = conn.sql("SELECT layer, X, Y FROM data WHERE name LIKE 'String_Tracker_%'")
print(structures)

┌─────────┬───────────┬───────────┐
│  layer  │     X     │     Y     │
│ varchar │   float   │   float   │
├─────────┼───────────┼───────────┤
│ MDS_1.1 │  511819.7 │ 8087402.5 │
│ MDS_1.1 │  511819.7 │ 8087368.5 │
│ MDS_1.1 │  511819.7 │ 8087334.5 │
│ MDS_1.1 │  511819.7 │ 8087230.5 │
│ MDS_1.1 │  511819.7 │ 8087264.5 │
│ MDS_1.1 │  511819.7 │ 8087298.5 │
│ MDS_1.1 │  511812.9 │ 8087402.5 │
│ MDS_1.1 │  511812.9 │ 8087368.5 │
│ MDS_1.1 │  511812.9 │ 8087334.5 │
│ MDS_1.1 │  511812.9 │ 8087230.5 │
│    ·    │      ·    │     ·     │
│    ·    │      ·    │     ·     │
│    ·    │      ·    │     ·     │
│ MDS_3.2 │  513069.7 │ 8088390.0 │
│ MDS_3.2 │  513076.5 │ 8088494.0 │
│ MDS_3.2 │  513076.5 │ 8088460.0 │
│ MDS_3.2 │  513076.5 │ 8088426.0 │
│ MDS_3.2 │  513076.5 │ 8088322.0 │
│ MDS_3.2 │  513076.5 │ 8088356.0 │
│ MDS_3.2 │  513076.5 │ 8088390.0 │
│ MDS_3.2 │ 513083.28 │ 8088494.0 │
│ MDS_3.2 │ 513083.28 │ 8088460.0 │
│ MDS_3.2 │ 513083.28 │ 8088426.0 │
└─────────┴───────────┴─────

In [20]:
structure_df = structures.fetchdf()
structure_df['Line'] = round(structure_df['X'],0)

structure_df.sort_values(by=['Line', 'Y'], ascending=[True, False], inplace=True, ignore_index=True)

#Different Lines
condition1 = structure_df["Line"] != structure_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = (abs(structure_df["Y"] - structure_df["Y"].shift()) > 37.5)


break_condition = condition1 | condition2

structure_df['Group'] = break_condition.cumsum()
print(f'The number of structures is: {max(structure_df['Group'])}')

number_of_structures = pd.DataFrame({'Number of Structures': max(structure_df['Group'])}, index=[0])
print(number_of_structures)

The number of structures is: 1765
   Number of Structures
0                  1765


CABLE DATA

In [21]:
ac_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

dc_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_vala_CC_%C'
""")

QTY. OF AC CABLES (m)
=

In [22]:
print(ac_cables)

┌───────────────────────┬──────────┬───────────┬───────────┬───────────┬───────────┐
│         layer         │ start_x  │  start_y  │   end_x   │   end_y   │  length   │
│        varchar        │  float   │   float   │   float   │   float   │   float   │
├───────────────────────┼──────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_bt_vala_btca_01xC │ 511676.9 │ 8087210.0 │  511676.9 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511704.1 │ 8087210.0 │  511704.1 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511724.5 │ 8087210.0 │  511724.5 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511744.9 │ 8087210.0 │  511744.9 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511772.1 │ 8087210.0 │  511772.1 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511792.5 │ 8087244.0 │  511792.5 │ 8087246.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511819.7 │ 8087210.0 │  511819.7 │ 8087212.0 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 511840.1 │ 8087210.0 │  511840.1 │ 8087

In [23]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌───────────────────────┬────────────────────┐
│         layer         │    total_length    │
│        varchar        │       double       │
├───────────────────────┼────────────────────┤
│ EMF_bt_vala_btca_01xC │  4410.168501138687 │
│ EMF_bt_vala_btca_02xC │  2456.823513031006 │
│ EMF_bt_vala_btca_03xC │ 2504.7156183719635 │
│ EMF_bt_vala_btca_04xC │ 2708.6301980018616 │
│ EMF_bt_vala_btca_05xC │  2627.770488023758 │
│ EMF_bt_vala_btca_06xC │ 2148.7090842723846 │
│ EMF_bt_vala_btca_07xC │ 2028.3700342178345 │
│ EMF_bt_vala_btca_08xC │ 1148.7599580287933 │
│ EMF_bt_vala_btca_09xC │ 1898.8686439990997 │
└───────────────────────┴────────────────────┘



QTY. OF DC CABLES (m)
=

In [24]:
print(dc_cables)

┌─────────────────────────┬──────────┬───────────┬───────────┬───────────┬───────────┐
│          layer          │ start_x  │  start_y  │   end_x   │   end_y   │  length   │
│         varchar         │  float   │   float   │   float   │   float   │   float   │
├─────────────────────────┼──────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_cabo_solar_positivo │ 511778.9 │ 8087280.5 │  511791.8 │ 8087246.0 │  45.90625 │
│ EMF_cabo_solar_positivo │ 511778.9 │ 8087246.5 │  511791.8 │ 8087246.0 │   16.0127 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087418.5 │  511791.8 │ 8087246.0 │   177.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087384.5 │  511791.8 │ 8087246.0 │   143.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087349.5 │  511791.8 │ 8087246.0 │   108.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087314.5 │  511791.8 │ 8087246.0 │    73.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087280.5 │  511791.8 │ 8087246.0 │    39.125 │
│ EMF_cabo_solar_positivo │ 511792.5 │ 8087

In [25]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬────────────────────┐
│          layer          │    total_length    │
│         varchar         │       double       │
├─────────────────────────┼────────────────────┤
│ EMF_cabo_solar_negativo │ 1383589.5215280056 │
│ EMF_cabo_solar_positivo │   1344889.08309412 │
└─────────────────────────┴────────────────────┘

The total length of DC cables is 2728478.605 meters


QTY. OF DC DUCTS (m and un.)
=

In [26]:
print(dc_ducts)

┌──────────────────────┬───────────┬───────────┬────────────┬───────────┬───────────┐
│        layer         │  start_x  │  start_y  │   end_x    │   end_y   │  length   │
│       varchar        │   float   │   float   │   float    │   float   │   float   │
├──────────────────────┼───────────┼───────────┼────────────┼───────────┼───────────┤
│ EMF_bt_eletroduto_1S │  511690.5 │ 8087214.5 │   511703.4 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511710.9 │ 8087214.5 │  511704.78 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511758.5 │ 8087214.5 │   511771.4 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511778.9 │ 8087214.5 │  511772.78 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511853.7 │ 8087214.5 │   511859.8 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511874.1 │ 8087214.5 │   511861.2 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511921.7 │ 8087214.5 │   511927.8 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511670.1 │ 8087193.5 │   511

In [27]:
longest_DC_cable = conn.sql('SELECT * FROM dc_cables ORDER BY length DESC LIMIT 1')
print(longest_DC_cable) #Longest DC cable

┌─────────────────────────┬──────────┬───────────┬──────────┬───────────┬──────────┐
│          layer          │ start_x  │  start_y  │  end_x   │   end_y   │  length  │
│         varchar         │  float   │   float   │  float   │   float   │  float   │
├─────────────────────────┼──────────┼───────────┼──────────┼───────────┼──────────┤
│ EMF_cabo_solar_positivo │ 512933.7 │ 8088408.5 │ 512940.5 │ 8088129.0 │ 284.6738 │
└─────────────────────────┴──────────┴───────────┴──────────┴───────────┴──────────┘



In [28]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬────────────────────┐
│        layer         │    total_length    │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ EMF_bt_eletroduto_1S │  3844.093280315399 │
│ EMF_bt_eletroduto_2S │  7557.504392147064 │
│ EMF_bt_eletroduto_3S │ 23894.632987499237 │
└──────────────────────┴────────────────────┘



In [29]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │  3719 │
│ EMF_bt_eletroduto_2S │   635 │
│ EMF_bt_eletroduto_1S │   341 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 4695


QTY. OF DC TRENCHES (m)
=

In [30]:
print(dc_trenches)

┌─────────────────┬────────────┬───────────┬────────────┬───────────┬────────────┐
│      layer      │  start_x   │  start_y  │   end_x    │   end_y   │   length   │
│     varchar     │   float    │   float   │   float    │   float   │   float    │
├─────────────────┼────────────┼───────────┼────────────┼───────────┼────────────┤
│ EMF_vala_CC_1C  │  512320.25 │ 8087250.5 │ 512317.625 │ 8087248.0 │      3.625 │
│ EMF_vala_CC_1C  │  512197.84 │ 8087196.0 │  512200.47 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512225.06 │ 8087196.0 │  512222.44 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512265.84 │ 8087196.0 │  512268.47 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512293.06 │ 8087196.0 │  512290.44 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512333.84 │ 8087196.0 │  512336.47 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512361.06 │ 8087196.0 │  512358.44 │ 8087198.5 │      3.625 │
│ EMF_vala_CC_1C  │  512401.84 │ 8087196.0 │  512404.47 │ 8087198.5 │      3.625 │
│ EM

In [31]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────────┬────────────────────┐
│      layer      │    total_length    │
│     varchar     │       double       │
├─────────────────┼────────────────────┤
│ EMF_vala_CC_10C │  846.3577001690865 │
│ EMF_vala_CC_11C │            3.46875 │
│ EMF_vala_CC_12C │            2.96875 │
│ EMF_vala_CC_13C │            3.46875 │
│ EMF_vala_CC_14C │                3.5 │
│ EMF_vala_CC_1C  │  258.3968994617462 │
│ EMF_vala_CC_2C  │  3441.166440963745 │
│ EMF_vala_CC_3C  │  310.8299083709717 │
│ EMF_vala_CC_4C  │ 2844.2146011590958 │
│ EMF_vala_CC_5C  │ 1769.1333198547363 │
│ EMF_vala_CC_6C  │  6942.574889063835 │
│ EMF_vala_CC_7C  │  113.8006477355957 │
│ EMF_vala_CC_8C  │ 1204.6789104938507 │
│ EMF_vala_CC_9C  │            87.1875 │
└─────────────────┴────────────────────┘
  14 rows                    2 columns


The total length of DC trenches is 17831.747 meters


OVERVIEW
=

In [32]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\Minas do Sol - BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)
    number_of_structures.to_excel(xlsx, sheet_name='Number of Structures', index=False)


conn.close()
